# 01 · Data exploration: Frangieh et al. 2021 Perturb-CITE-seq

Goal: understand what is actually in the data before choosing a model, and check the paper's claims against it, one section at a time.

**Section 1: what files exist, and what is in the processed one.**

In [1]:
import pandas as pd
import requests

from tmesim.data import DATA, load_adata

pd.set_option("display.max_columns", 50)
DATA

PosixPath('/Users/shreyan/Downloads/TMEsim/data')

## 1.1 Which Frangieh files does pertpy provide?

pertpy ships four versions of this dataset. Before downloading anything else, list their sizes and which ones are already local.

In [2]:
BASE_URL = "https://scverse-exampledata.s3.eu-west-1.amazonaws.com/pertpy"
variants = ["frangieh_2021", "frangieh_2021_rna", "frangieh_2021_protein", "frangieh_2021_raw"]

rows = []
for name in variants:
    size = int(requests.head(f"{BASE_URL}/{name}.h5ad", timeout=30).headers["content-length"])
    rows.append({"file": f"{name}.h5ad", "size_GB": round(size / 1e9, 2), "local": (DATA / f"{name}.h5ad").exists()})
pd.DataFrame(rows)

,file,size_GB,local
0,frangieh_2021.h5ad,1.04,True
1,frangieh_2021_rna.h5ad,1.46,False
2,frangieh_2021_protein.h5ad,0.02,False
3,frangieh_2021_raw.h5ad,6.16,False


## 1.2 Load the processed dataset

`frangieh_2021` is the processed version and the one already downloaded; `load_adata()` reuses `data/frangieh_2021.h5ad`.

In [3]:
adata = load_adata()
adata

AnnData object with n_obs × n_vars = 126966 × 22097
    obs: 'library_preparation_protocol', 'condition', 'MOI', 'sgRNA', 'UMI_count', 'n_genes', 'leiden', 'perturbation_name', 'perturbation_type', 'perturbation_value', 'perturbation_unit'
    var: 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'MOI_colors', 'condition_colors', 'doi', 'hvg', 'leiden', 'leiden_colors', 'library_preparation_protocol_colors', 'neighbors', 'pca', 'perturbation_type_colors', 'perturbation_unit_colors', 'perturbation_value_colors', 'preprocessing_nb_link', 'protein_names', 'umap'
    obsm: 'X_pca', 'X_umap', 'protein_expression'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'

## 1.3 Paper claims to check

Claims from Frangieh et al. 2021 (main text and Methods) that this data can confirm or refute. Later sections fill in the last column.

In [4]:
claims = pd.DataFrame(
    [
        ("High-quality cells analyzed", "218,331"),
        ("Cells per condition", "co-culture 73,114 · IFNγ 87,590 · control 57,627"),
        ("Contaminating T cells removed", "805"),
        ("Library", "744 targeting sgRNAs, 248 genes, plus non-targeting controls"),
        ("Surface proteins", "20"),
        ("Transduction", "MOI 0.1 (mostly single-guide cells)"),
        ("RNA normalization", "counts per 1e6, then natural log"),
        ("Protein normalization", "max(ln((c_antibody + 1) / (c_IgG + 1)), 0)"),
        ("IFNγ arm", "1 ng/mL for 16 h"),
        ("Co-culture arm", "TIL:tumor 2:1 for 48 h, after 16 h IFNγ pretreatment"),
    ],
    columns=["claim", "paper"],
)
claims["in this data"] = "not checked yet"
claims

,claim,paper,in this data
0,High-quality cells analyzed,"218,331",not checked yet
1,Cells per condition,"co-culture 73,114 · IFNγ 87,590 · control 57,627",not checked yet
2,Contaminating T cells removed,805,not checked yet
3,Library,"744 targeting sgRNAs, 248 genes, plus non-targ...",not checked yet
4,Surface proteins,20,not checked yet
5,Transduction,MOI 0.1 (mostly single-guide cells),not checked yet
6,RNA normalization,"counts per 1e6, then natural log",not checked yet
7,Protein normalization,"max(ln((c_antibody + 1) / (c_IgG + 1)), 0)",not checked yet
8,IFNγ arm,1 ng/mL for 16 h,not checked yet
9,Co-culture arm,"TIL:tumor 2:1 for 48 h, after 16 h IFNγ pretre...",not checked yet


## 1.4 First observations

- `obs` has 11 per-cell columns; `obsm` holds PCA, UMAP and a 20-column `protein_expression` matrix; there are no `layers` and no `raw`.
- The cell count (126,966) is already well below the paper's 218,331. Section 2 looks at the cell metadata to see why.